In [1]:
import pandas as pd
import numpy as np

In [2]:
temp_df=pd.read_csv('/IMDB Dataset.csv')

In [3]:
df= temp_df.iloc[:10000]

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df['review'][1]

'A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master\'s of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell\'s murals decorating every surface) are terribly well d

In [6]:
df['sentiment'].value_counts()

,count
sentiment,
positive,5028
negative,4972


In [7]:
df.isnull().sum()

,0
review,0
sentiment,0


In [8]:
df.duplicated().sum()

np.int64(17)

In [9]:
df.drop_duplicates(inplace= True)

/tmp/ipykernel_15304/2207841240.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace= True)


In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
import re
def remove_tags(raw_text):
    cleaned_text= re.sub(re.compile('<.*?>'),'',raw_text)
    return cleaned_text

In [12]:
df['review'] = df['review'].apply(remove_tags)

/tmp/ipykernel_15304/2336150696.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(remove_tags)


In [13]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
9995,"Fun, entertaining movie about WWII German spy ...",positive
9996,Give me a break. How can anyone say that this ...,negative
9997,This movie is a bad movie. But after watching ...,negative
9998,This is a movie that was probably made to ente...,negative


In [14]:
df['review'] = df['review'].apply(lambda x:x.lower())

/tmp/ipykernel_15304/740760900.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(lambda x:x.lower())


In [15]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
sw_list = stopwords.words('english')
df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
/tmp/ipykernel_15304/980428484.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))


In [16]:
df['review']

,review
0,one reviewers mentioned watching 1 oz episode ...
1,wonderful little production. filming technique...
2,thought wonderful way spend time hot summer we...
3,basically there's family little boy (jake) thi...
4,"petter mattei's ""love time money"" visually stu..."
...,...
9995,"fun, entertaining movie wwii german spy (julie..."
9996,"give break. anyone say ""good hockey movie""? kn..."
9997,movie bad movie. watching endless series bad h...
9998,"movie probably made entertain middle school, e..."


In [18]:
!pip install gensim
import gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 40.6 MB/s  0:00:00


In [19]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [21]:
import nltk
nltk.download('punkt_tab')

story=[]
for doc in df['review']:
  raw_sent =sent_tokenize(doc)
  for sent in raw_sent:
    story.append(simple_preprocess(sent))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [22]:
model =gensim.models.Word2Vec(
    window=10,
    min_count=2
)

In [23]:
model.build_vocab(story)

In [24]:
model.train(story, total_examples= model.corpus_count, epochs= model.epochs)

(5849569, 6186875)

In [25]:
len(model.wv.index_to_key)

31845

In [26]:
from gensim.models import word2vec
def document_vector(doc):
  #remove out_of_vocabulary words
  doc = [word for word in doc.split() if word in model.wv.index_to_key]
  return np.mean(model.wv[doc], axis=0)

In [27]:
document_vector(df['review'].values[0])

array([-0.02815614,  0.38427144,  0.1794117 , -0.09033418, -0.02902945,
       -0.5142546 ,  0.19379215,  0.8218999 , -0.2551434 , -0.0597799 ,
       -0.03876837, -0.5254096 ,  0.05117264,  0.16616   ,  0.05855098,
       -0.19353274,  0.12333898, -0.5707822 , -0.00099744, -0.70102364,
        0.1795992 ,  0.07468677,  0.3023454 , -0.2102301 , -0.18576002,
       -0.13857463, -0.26227927, -0.272224  , -0.3302459 ,  0.05605106,
        0.40971494,  0.08929387,  0.06659043, -0.27713495, -0.23120058,
        0.23315242,  0.04271089, -0.40202114, -0.17986086, -0.7335356 ,
        0.16819312, -0.28320855, -0.17978595,  0.08257158,  0.4202337 ,
       -0.20979191, -0.39074248, -0.13495013,  0.20168266,  0.27522373,
        0.13851961, -0.28175533, -0.17536265, -0.159105  , -0.43683177,
        0.09024284,  0.3359605 ,  0.12513043, -0.32608247,  0.09017982,
        0.01974968,  0.1244069 , -0.16038579,  0.05179555, -0.41989815,
        0.3595811 ,  0.0141908 ,  0.19362703, -0.61859095,  0.24

In [28]:
from tqdm import tqdm

In [29]:
X= []
for doc in tqdm(df['review'].values):
  X.append(document_vector(doc))

100%|██████████| 9983/9983 [10:50<00:00, 15.35it/s]


In [30]:
X= np.array(X)

In [31]:
X.shape

(9983, 100)

In [32]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y= encoder.fit_transform(df['sentiment'])

In [33]:
y

array([1, 1, 1, ..., 0, 0, 1], shape=(9983,))

In [34]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [35]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test, y_pred)

0.771657486229344